# Subspace Methods Deep Dive
## Tutorial 10: Forward-Backward Averaging, Spatial Smoothing, and Correlated Sources

Real-world signals are often **correlated or coherent**.  This breaks the rank assumption in MUSIC and ESPRIT.  We study:

1. **Effect of correlation** on subspace methods
2. **Forward-backward (FB) averaging** — when it helps
3. **Spatial smoothing** — breaking coherence
4. **Subspace perturbation theory** — why performance degrades
5. **Combined techniques**

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, os

sys.path.insert(0, os.path.join(os.getcwd(), '..', 'src'))

from doa_methods.array_processing import UniformLinearArray, SignalModel
from doa_methods.subspace import MUSIC, ESPRIT

plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

M = 16
array  = UniformLinearArray(M=M, d=0.5)
sm     = SignalModel(array)
music  = MUSIC(array)
esprit = ESPRIT(array)

doas_true = np.deg2rad([-20.0, 15.0])
K = len(doas_true)

angle_grid = np.linspace(-np.pi/2, np.pi/2, 3601)

print("Setup complete.")

## 1. Effect of Source Correlation

The theoretical covariance matrix $\mathbf{R} = \mathbf{A}\mathbf{P}_s\mathbf{A}^H + \sigma_n^2\mathbf{I}$ assumes **uncorrelated** sources ($\mathbf{P}_s$ diagonal).

With **correlated** sources, $\mathbf{P}_s$ is no longer diagonal and can be rank-deficient for **fully coherent** (same waveform) sources.  When sources are coherent, the signal subspace **collapses** — the effective rank of $\mathbf{R}-\sigma_n^2\mathbf{I}$ drops below $K$.

In [ ]:
snr_db = 15
N = 200

correlations = [0.0, 0.5, 0.9, 1.0]
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

for ax, rho in zip(axes.flatten(), correlations):
    X_c, _, _ = sm.generate_signals(
        doas_true, N, snr_db, correlation=rho, seed=7)
    R_c = X_c @ X_c.conj().T / N
    evals, _ = np.linalg.eigh(R_c)
    evals = evals[::-1]

    spec_c = np.zeros(len(angle_grid))
    evals_c, evecs_c = np.linalg.eigh(R_c)
    evals_c = evals_c[::-1]; evecs_c = evecs_c[:, ::-1]
    U_n_c = evecs_c[:, K:]
    A_g = array.array_manifold(angle_grid)
    proj_c = np.sum(np.abs(A_g.conj().T @ U_n_c)**2, axis=1)
    spec_c = 1/(proj_c + 1e-12)
    spec_c = 10*np.log10(spec_c / spec_c.max() + 1e-12)

    ax.plot(np.rad2deg(angle_grid), spec_c, 'b-', lw=2)
    for th in doas_true:
        ax.axvline(np.rad2deg(th), color='r', ls='--')
    rank_eff = np.sum(evals > 0.1*evals[0])
    ax.set_title(f'ρ = {rho}  (eff. rank ≈ {rank_eff})')
    ax.set_xlabel('θ (°)'); ax.set_ylim(-40, 2)
    ax.grid(True, alpha=0.3)

plt.suptitle('MUSIC Spectrum vs Source Correlation', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 2. Forward-Backward Averaging

For a ULA with the **centro-Hermitian** property, we can augment the sample covariance:

$$\hat{\mathbf{R}}_{\text{FB}} = \frac{1}{2}\left(\hat{\mathbf{R}} + \mathbf{J}\hat{\mathbf{R}}^*\mathbf{J}\right)$$

where $\mathbf{J}$ is the $M\times M$ exchange matrix (anti-diagonal identity).

**Effect**: effectively doubles the number of virtual snapshots.  Helps with limited $N$ and **partially** correlated sources.  For **fully coherent** sources, additional spatial smoothing is needed.

In [ ]:
from doa_methods.utils.math_utils import forward_backward_averaging

def music_spec_fb(X, K, use_fb):
    R_ = X @ X.conj().T / X.shape[1]
    if use_fb:
        R_ = forward_backward_averaging(R_)
    evals_, evecs_ = np.linalg.eigh(R_)
    U_n_ = evecs_[:, ::-1][:, K:]
    A_ = array.array_manifold(angle_grid)
    p = np.sum(np.abs(A_.conj().T @ U_n_)**2, axis=1)
    spec = 1/(p + 1e-12)
    return 10*np.log10(spec / spec.max() + 1e-12)

rho = 0.9    # high correlation
X_corr, _, _ = sm.generate_signals(doas_true, N, snr_db, correlation=rho, seed=42)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), music_spec_fb(X_corr, K, False),
        'b-', lw=2, label='MUSIC (no FB)')
ax.plot(np.rad2deg(angle_grid), music_spec_fb(X_corr, K, True),
        'r-', lw=2, label='MUSIC + FB averaging')
for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.6)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Normalised Spectrum (dB)')
ax.set_title(f'FB Averaging for Correlated Sources  (ρ={rho})')
ax.set_ylim(-40, 2); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 3. Spatial Smoothing

For **fully coherent** sources (multipath), FB averaging alone is insufficient.  **Spatial smoothing** (Shan et al., 1985) divides the array into $L$ overlapping subarrays of length $M_s = M - L + 1$, and averages their covariance matrices:

$$\hat{\mathbf{R}}_{\text{ss}} = \frac{1}{L}\sum_{l=1}^{L} \hat{\mathbf{R}}_l$$

where $\hat{\mathbf{R}}_l$ is the sample covariance of the $l$-th subarray.

**Key tradeoff**: more subarrays ($L\uparrow$) → better decorrelation, but reduced effective aperture ($M_s = M - L + 1$ ↓$).  Typically $L = K$ is a reasonable choice.

In [ ]:
def spatial_smoothing(X, L):
    """Forward spatial smoothing: average over L subarrays of length M_s = M-L+1."""
    M_full = X.shape[0]
    M_s = M_full - L + 1
    R_ss = np.zeros((M_s, M_s), dtype=complex)
    N_ = X.shape[1]
    for l in range(L):
        X_l = X[l:l+M_s, :]
        R_ss += X_l @ X_l.conj().T / N_
    return R_ss / L, M_s

def music_with_smoothing(X, K_true, L):
    """MUSIC on a spatially smoothed covariance."""
    R_ss, M_s = spatial_smoothing(X, L)
    evals_, evecs_ = np.linalg.eigh(R_ss)
    evals_ = evals_[::-1]; evecs_ = evecs_[:, ::-1]
    U_n_ = evecs_[:, K_true:]
    # Use the first M_s elements of the angle grid manifold
    arr_small = UniformLinearArray(M=M_s, d=0.5)
    A_ = arr_small.array_manifold(angle_grid)
    p = np.sum(np.abs(A_.conj().T @ U_n_)**2, axis=1)
    spec = 1/(p+1e-12)
    return 10*np.log10(spec/spec.max()+1e-12)

from doa_methods.array_processing import UniformLinearArray

# Fully coherent sources (ρ=1)
X_coh, _, _ = sm.generate_signals(doas_true, N, snr_db, correlation=1.0, seed=5)

fig, ax = plt.subplots(figsize=(13, 6))
ax.plot(np.rad2deg(angle_grid), music_spec_fb(X_coh, K, False),
        'b--', lw=1.5, label='MUSIC (no processing)')
ax.plot(np.rad2deg(angle_grid), music_spec_fb(X_coh, K, True),
        'g-',  lw=1.5, label='MUSIC + FB avg')

for L in [2, K, 4]:
    spec_ss = music_with_smoothing(X_coh, K, L)
    ax.plot(np.rad2deg(angle_grid), spec_ss, lw=2,
            label=f'Spatial smoothing L={L}  (M_s={M-L+1})')

for th in doas_true:
    ax.axvline(np.rad2deg(th), color='k', ls='--', alpha=0.5)
ax.set_xlabel('θ (°)'); ax.set_ylabel('Normalised Spectrum (dB)')
ax.set_title('Spatial Smoothing for Coherent Sources  (ρ=1)')
ax.set_ylim(-40, 2); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 4. Effect of Smoothing on Aperture and Resolution

More smoothing → better decorrelation but smaller effective aperture.

In [ ]:
L_values = [1, 2, 4, 8]
X_coh2, _, _ = sm.generate_signals(doas_true, 300, snr_db=20, correlation=1.0, seed=9)

fig, axes = plt.subplots(1, len(L_values), figsize=(18, 5), sharey=True)
for ax, L in zip(axes, L_values):
    M_s = M - L + 1
    spec_ss = music_with_smoothing(X_coh2, K, L)
    ax.plot(np.rad2deg(angle_grid), spec_ss, 'b-', lw=2)
    for th in doas_true:
        ax.axvline(np.rad2deg(th), color='r', ls='--')
    ax.set_title(f'L={L}  M_s={M_s}')
    ax.set_xlabel('θ (°)'); ax.set_ylim(-40, 2)
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Normalised Spectrum (dB)')
plt.suptitle('Aperture–Decorrelation Tradeoff in Spatial Smoothing', fontsize=13, fontweight='bold')
plt.tight_layout(); plt.show()

## 5. Practical Recommendations

| Scenario | Recommended Technique |
|---|---|
| Uncorrelated sources, ample snapshots | Standard MUSIC / ESPRIT |
| Uncorrelated sources, few snapshots | MUSIC + FB averaging |
| Partially correlated ($\rho < 0.8$) | MUSIC + FB averaging |
| Fully coherent (multipath) | Spatial smoothing ($L \geq K$) + optional FB |
| Coherent + limited snapshots | FB + spatial smoothing |

**Rule of thumb**: use $L = K$ subarrays for spatial smoothing, giving $M_s = M - K + 1$ effective elements.

## Summary

- Correlation degrades MUSIC/ESPRIT because it collapses the signal-subspace rank.
- FB averaging helps for partial correlation and limited snapshots at no aperture cost.
- Spatial smoothing restores rank for coherent sources but reduces aperture.
- For $M=16, K=2$: spatial smoothing with $L=2$ loses 1 element — an acceptable tradeoff.

## Exercises
1. Derive analytically why the spatial smoothed covariance matrix has higher rank than the original for coherent sources (hint: consider the phase differences between subarrays).
2. Implement **forward-backward spatial smoothing** (combine FB averaging with spatial smoothing) and compare with forward-only smoothing.
3. Monte Carlo: for $M=16, K=2, \text{SNR}=15$ dB, $N=200$, plot RMSE vs $\rho \in [0, 1]$ for plain MUSIC, MUSIC+FB, and MUSIC+SS(L=2).